# Model 5: U + I + Content + Popularity + Time Decay

**Precision@10:** 0.0555 | **Recall@10:** 0.2940

Two new ideas are added on top of Model 4:
1. **Time decay** — recent borrowings matter more than old ones
2. **Popularity** — a gentle tiebreaker for trending books

## Step 1: Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')


## Step 2: Load Data

In [ ]:
interactions = pd.read_csv('../data/interactions_train.csv').rename(
    columns={'u': 'user_id', 'i': 'book_id', 't': 'timestamp'})
books = pd.read_csv('../data/items.csv')

user_id_map = {o: n for n, o in enumerate(interactions['user_id'].unique())}
book_id_map = {o: n for n, o in enumerate(interactions['book_id'].unique())}
interactions['user_id'] = interactions['user_id'].map(user_id_map)
interactions['book_id'] = interactions['book_id'].map(book_id_map)

n_users = interactions['user_id'].nunique()
n_items = interactions['book_id'].nunique()
print(f'Users: {n_users} | Books: {n_items} | Interactions: {len(interactions)}')


## Step 3: Helper Functions

### Time-Decay Weighted Matrix

Instead of a binary matrix, each borrowing event gets a weight based on its rank in the user's borrowing history. The most recent book gets weight **1.0**; older books decay by a factor of **(1 - 0.03)** per step.

For a user with 3 books (oldest → newest): **0.94 → 0.97 → 1.00**

In [ ]:
def create_weighted_matrix(data, n_users, n_items, decay=0.03):
    """Rank-based time-decay: most recent borrowing = weight 1.0."""
    mat = np.zeros((n_users, n_items))
    for _, ud in data.groupby('user_id'):
        ud = ud.sort_values('timestamp')
        n  = len(ud)
        w  = np.array([(1 - decay) ** (n - 1 - i) for i in range(n)])
        w /= w.max()
        for i, (_, row) in enumerate(ud.iterrows()):
            mat[int(row['user_id']), int(row['book_id'])] = w[i]
    return mat

def create_data_matrix(data, n_users, n_items):
    mat = np.zeros((n_users, n_items))
    mat[data['user_id'].values, data['book_id'].values] = 1
    return mat

def user_based_predict(mat, sim, eps=1e-9):
    return sim.dot(mat) / (np.abs(sim).sum(axis=1)[:, None] + eps)

def item_based_predict(mat, sim, eps=1e-9):
    return (sim.dot(mat.T) / (sim.sum(axis=1)[:, None] + eps)).T

def normalize(mat):
    lo, hi = mat.min(), mat.max()
    return (mat - lo) / (hi - lo + 1e-9)

def precision_recall_at_k(scores, gt, k=10):
    n = scores.shape[0]
    total_p, total_r = 0.0, 0.0
    for u in range(n):
        true_items = np.where(gt[u] == 1)[0]
        top_k = np.argsort(scores[u])[-k:]
        hits  = np.isin(top_k, true_items).sum()
        total_p += hits / k
        total_r += hits / len(true_items) if len(true_items) > 0 else 0
    return total_p / n, total_r / n


## Step 4: TF-IDF with Author Emphasis

We double the author field in the text representation so that books by the same author are ranked closer to each other.

In [ ]:
for col in ['Title', 'Author', 'Subjects', 'Publisher']:
    books[col] = books[col].fillna('')
books['text'] = (books['Title'] + ' ' + books['Author'] + ' ' + books['Author'] + ' ' +
                 books['Subjects'] + ' ' + books['Subjects'] + ' ' + books['Publisher'])

tfidf = TfidfVectorizer(max_features=10000, strip_accents='unicode', min_df=2)
tfidf_matrix = tfidf.fit_transform(books['text'])
mapping = {book_id_map[o]: i for i, o in enumerate(books['i']) if o in book_id_map}
books_i = books['i'].values
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')


## Step 5: 5-Fold Cross-Validation

**Final score = 0.75 × CF_decay + 0.20 × Content + 0.05 × Popularity**

Popularity is log-normalized to prevent bestsellers from drowning out personalised matches.

In [ ]:
interactions_s = interactions.sort_values(['user_id', 'timestamp']).copy()
interactions_s['fold'] = interactions_s.groupby('user_id')['timestamp'].transform(
    lambda x: pd.qcut(x.rank(method='first'), 5, labels=False))

precisions, recalls = [], []

for fold in range(5):
    print(f'Fold {fold+1}/5...')
    train = interactions_s[interactions_s['fold'] != fold]
    test  = interactions_s[interactions_s['fold'] == fold]

    train_mat = create_weighted_matrix(train, n_users, n_items)
    test_mat  = np.zeros((n_users, n_items))
    test_mat[test['user_id'].values, test['book_id'].values] = 1

    # CF with time-decay weights
    cf = 0.45 * normalize(user_based_predict(train_mat, cosine_similarity(train_mat))) + \
         0.55 * normalize(item_based_predict(train_mat, cosine_similarity(train_mat.T)))

    # Content (simple mean profile)
    user_books = train.groupby('user_id')['book_id'].apply(list).to_dict()
    content = np.zeros((n_users, n_items))
    for u in range(n_users):
        rows = [mapping[b] for b in user_books.get(u, []) if b in mapping]
        if not rows: continue
        profile = np.asarray(tfidf_matrix[rows].mean(axis=0))
        scores  = cosine_similarity(profile, tfidf_matrix).flatten()
        for idx, orig in enumerate(books_i):
            if orig in book_id_map:
                content[u, book_id_map[orig]] = scores[idx]
    content = normalize(content)

    # Popularity (log-normalized unique readers per book)
    binary_mat = create_data_matrix(train, n_users, n_items)
    pop = normalize(np.log1p(binary_mat.sum(axis=0)))

    hybrid = 0.75 * cf + 0.20 * content + 0.05 * pop
    p, r = precision_recall_at_k(hybrid, test_mat)
    precisions.append(p); recalls.append(r)
    print(f'  Precision@10: {p:.4f} | Recall@10: {r:.4f}')


## Results

In [ ]:
print('=' * 45)
print('AVERAGE RESULTS (5-FOLD CV):')
print(f'  Precision@10: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}')
print(f'  Recall@10:    {np.mean(recalls):.4f} ± {np.std(recalls):.4f}')
print('=' * 45)
for i, (p, r) in enumerate(zip(precisions, recalls)):
    print(f'  Fold {i+1}: P={p:.4f}  R={r:.4f}')


## Key Takeaways

- **Time decay** lifts Precision@10 from 0.0532 → ~0.0545: recent borrowings are a stronger signal of current taste than old ones.
- **Popularity** provides a small additional lift as a tiebreaker, but is capped at 5% weight to avoid turning recommendations into a bestseller list.
- Overall improvement vs Model 4: **+0.0023**

## Step 6: Generate Submission

Now that we have validated the model, we train on **100% of the data** (no holdout) and generate the final recommendation file for Kaggle.

In [ ]:
# Train on full dataset
full_mat = create_weighted_matrix(interactions, n_users, n_items)

cf_full = 0.45 * normalize(user_based_predict(full_mat, cosine_similarity(full_mat))) + \
          0.55 * normalize(item_based_predict(full_mat, cosine_similarity(full_mat.T)))

user_books_full = interactions.groupby('user_id')['book_id'].apply(list).to_dict()
content_full = np.zeros((n_users, n_items))
for u in range(n_users):
    rows = [mapping[b] for b in user_books_full.get(u, []) if b in mapping]
    if not rows: continue
    profile = np.asarray(tfidf_matrix[rows].mean(axis=0))
    scores  = cosine_similarity(profile, tfidf_matrix).flatten()
    for idx, orig in enumerate(books_i):
        if orig in book_id_map:
            content_full[u, book_id_map[orig]] = scores[idx]
content_full = normalize(content_full)

binary_full = create_data_matrix(interactions, n_users, n_items)
pop_full    = normalize(np.log1p(binary_full.sum(axis=0)))

final_scores = 0.75 * cf_full + 0.20 * content_full + 0.05 * pop_full
print('Full-data scoring complete.')


In [ ]:
# Save submission CSV
K = 10
user_id_inverse = {v: k for k, v in user_id_map.items()}
book_id_inverse = {v: k for k, v in book_id_map.items()}

with open('submission_model5.csv', 'w') as f:
    f.write('user_id,recommendation\n')
    for u in range(n_users):
        top_k = np.argsort(final_scores[u])[::-1][:K]
        books_str = ' '.join(str(book_id_inverse[i]) for i in top_k)
        f.write(f'{user_id_inverse[u]},{books_str}\n')

print('Saved submission_model5.csv')
